<a href="https://colab.research.google.com/github/Renata-Nascimento/Projeto-U02-CD/blob/main/notebooks/01_coleta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
  <img src="https://www.ufrn.br/resources/img/logo-ufrn.png" alt="Logo UFRN" width="150">
</p>

Este caderno corresponde ao projeto da 2ª unidade da disciplina Ciência de Dados.

Membros:
*   Brenda Gomes
*   Fernanda Nascimento
*   Gabriel Borges
*   Renata Nascimento

**Tema: Para onde vai o dinheiro do Estado nos municípios do RN?**

**Problema:** o Governo do RN repassa aos 167 municípios uma parte dos impostos que arrecada (cota-parte do ICMS, do IPVA, do IPI-Exportação e dos royalties do petróleo). Esses repasses são obrigatórios e seguem regras definidas em lei, que levam em conta principalmente a atividade econômica de cada município. Queremos investigar se essa distribuição reduz ou reforça as desigualdades entre os municípios: o dinheiro chega mais a quem é rico ou a quem é pobre? E isso mudou ao longo do tempo?

**Perguntas de pesquisa:**

* Quão desigual é a distribuição dos repasses por habitante entre os municípios, e essa desigualdade mudou entre 2020 e 2023?
* Municípios mais pobres (menor PIB per capita, maior proporção de pessoas de baixa renda no CadÚnico) recebem mais ou menos por habitante?
* Há concentração dos repasses em determinadas regiões ou portes de município?

**Período:** 2020 a 2023, o maior intervalo em que todas as fontes têm dado publicado (o PIB municipal só vai até 2023). Os valores foram corrigidos pela inflação (IPCA, a preços de 2023).

**Fontes (integradas pelo código IBGE do município):**

* SEFAZ/RN – Transferências para os Municípios: repasses mensais de ICMS, IPVA, IPI e royalties.
* IBGE (API): lista de municípios e regiões, população (Censo 2022), PIB municipal e IPCA.
* MDS – CadÚnico (API VIS DATA): pessoas cadastradas e de baixa renda por município.

**Fora do escopo nesta etapa:** convênios, emendas parlamentares e IDHM. Os convênios representaram no máximo 2% do valor repassado por ano, as emendas só estão disponíveis em PDFs sem identificação padronizada do município, e o IDHM municipal mais recente é de 2010. Ficam como possível extensão do trabalho.

Para esta semana (13/10 - 16/10):
- código com a coleta de dados

- dicionário de dados

- avaliação da viabilidade e da qualidade dos dados obtidos

- principais conclusões e aprendizados até o momento

# CP1 — Coleta e integração dos dados

**Pergunta central:** os repasses obrigatórios do Governo do RN (ICMS, IPVA, IPI, Royalties) chegam mais a
municípios ricos ou pobres? E isso mudou ao longo do tempo?

**Recorte:** 167 municípios do RN · período **2020 a 2023**.

| # | Fonte | O que traz | Como coletamos |
|---|-------|-----------|----------------|
| 1 | SEFAZ/RN | Repasses mensais por município (ICMS, IPVA, IPI, Royalties) | Planilhas `.xls` (1 por ano) |
| 2 | IBGE | Municípios e regiões, população (Censo 2022), PIB municipal e IPCA | API |
| 3 | MDS – CadÚnico | Pessoas cadastradas e de baixa renda (dezembro de cada ano) | API |

A chave que liga as fontes é o **código IBGE do município**.

### Por que o período 2020–2023?

Escolhemos o **maior intervalo em que todas as fontes têm dado publicado para todos os anos**:

| Fonte | Disponibilidade | Limita o período? |
|---|---|---|
| Repasses (SEFAZ) | 2020 a 2025 | não |
| CadÚnico (MDS) | mensal, até 2026 | não |
| **PIB municipal (IBGE)** | **até 2023** (o IBGE publica com cerca de 2 anos de atraso) | **sim → fim em 2023** |
| População (IBGE) | estimativas 2020, 2021, 2024, 2025 + Censo 2022; **sem dado para 2023** | resolvido abaixo |

- **Fim em 2023:** é o último ano com PIB municipal. Incluir 2024 e 2025 obrigaria a repetir o PIB de 2023,
  e a pergunta "ricos × pobres" ficaria sem medida de riqueza atual.
- **Início em 2020:** quatro anos permitem ver tendência sem misturar períodos muito diferentes. 2020 também mostra
  o efeito da pandemia na arrecadação, o que é útil para interpretar a série.

Dois cuidados para que a comparação entre anos seja justa:
1. **População fixa no Censo 2022 para todos os anos.** O Censo 2022 ficou cerca de 7% abaixo das estimativas
   anteriores. Se usássemos estimativa em 2020–2021 e Censo em 2022, o valor por habitante "pularia" em 2022 só
   por causa da troca de fonte. Com a população fixa, a variação per capita reflete só a variação do dinheiro.
   Além disso, 2023 não tem população publicada.
2. **Valores corrigidos pela inflação (IPCA), a preços de 2023.** O IPCA subiu cerca de 24% entre 2020 e 2023.
   Sem correção, os repasses pareceriam crescer bem mais do que realmente cresceram.

## 1. Bibliotecas e parâmetros

In [ ]:
import os, re, sys, glob, unicodedata
import numpy as np
import pandas as pd
import requests

ANOS = [2020, 2021, 2022, 2023]   # período da análise (justificativa acima)
ANO_PRECOS = 2023                # valores em R$ de 2023
os.makedirs('dados', exist_ok=True)
os.makedirs('saida', exist_ok=True)

## 2. Lista oficial de municípios (IBGE)

Baixamos primeiro porque ela é a "régua": todo nome de município das outras fontes é convertido para o código IBGE.

In [ ]:
resp = requests.get('https://servicodados.ibge.gov.br/api/v1/localidades/estados/24/municipios', timeout=60)

municipios = pd.DataFrame([{
    'cod_ibge': m['id'],
    'municipio': m['nome'],
    'mesorregiao': m['microrregiao']['mesorregiao']['nome'],
    'regiao_intermediaria': m['regiao-imediata']['regiao-intermediaria']['nome'],
} for m in resp.json()])

print(len(municipios), 'municípios')
municipios.head()

167 municípios


,cod_ibge,municipio,mesorregiao,regiao_intermediaria
0,2400109,Acari,Central Potiguar,Caicó
1,2400208,Assú,Oeste Potiguar,Mossoró
2,2400307,Afonso Bezerra,Central Potiguar,Mossoró
3,2400406,Água Nova,Oeste Potiguar,Mossoró
4,2400505,Alexandria,Oeste Potiguar,Mossoró


### Padronização dos nomes

As planilhas da SEFAZ não têm código IBGE, só o nome, e escrevem o mesmo município de jeitos diferentes
(`SEN ELOI DE SOUZA` × `SENADOR ELOI DE SOUZA`, `LAGOA D'ANTA` × `LAGOA DANTA`).
Criamos uma **chave** (só letras, sem acento e sem espaço) e um **de-para** para nomes abreviados ou antigos.

In [ ]:
def chave(nome):
    """'Olho d'Água do Borges' -> 'OLHODAGUADOBORGES'"""
    s = unicodedata.normalize('NFKD', str(nome)).encode('ascii', 'ignore').decode().upper()
    return re.sub(r'[^A-Z]', '', s)

DE_PARA = {
    'ARES': 'AREZ',
    'BOASAUDE': 'JANUARIOCICCO',                 # nome popular (oficial: Januário Cicco)
    'CAICARADORIODOSVENTOS': 'CAICARADORIODOVENTO',
    'GOVDIXSEPTROSADO': 'GOVERNADORDIXSEPTROSADO',
    'IPANGUASSU': 'IPANGUACU',
    'OLHODAGUADOSBORGES': 'OLHODAGUADOBORGES',
    'RUIBARBOSA': 'RUYBARBOSA',
    'SAOMIGUELDETOUROS': 'SAOMIGUELDOGOSTOSO',   # nome antigo
    'SENELOIDESOUZA': 'SENADORELOIDESOUZA',
    'SENGEORGINOAVELINO': 'SENADORGEORGINOAVELINO',
}

COD_POR_CHAVE = dict(zip(municipios['municipio'].map(chave), municipios['cod_ibge']))

## 3. Repasses obrigatórios (SEFAZ/RN)

Fonte: site da SEFAZ/RN → Transparência → Transferências para os Municípios do RN.
As planilhas `.xls` (uma por ano) estão guardadas no repositório do grupo, na pasta `dados/sefaz`,
e a célula abaixo baixa todas automaticamente. Anos fora do período (2024, 2025) são lidos,
mas ficam fora da análise.

In [1]:
# @title
#desativado - NÃO EXECUTAR
# if not glob.glob('dados/*.xls') and 'google.colab' in sys.modules:
#     from google.colab import files
#     for nome, conteudo in files.upload().items():
#         open(os.path.join('dados', nome), 'wb').write(conteudo)

# arquivos = sorted(glob.glob('dados/*.xls'))
# print(len(arquivos), 'arquivos')

NameError: name 'glob' is not defined

In [ ]:
# As planilhas da SEFAZ ficam no repositório do grupo, na pasta dados/sefaz.
# Em vez de cada pessoa fazer upload manual no Colab, o código:
#   1) pergunta ao GitHub quais arquivos existem nessa pasta (API de conteúdo do GitHub);
#   2) baixa cada arquivo .xls para uma pasta local do Colab com o mesmo nome (dados/sefaz).
# Assim qualquer colega roda o notebook sem precisar ter as planilhas no computador.
# Para incluir um ano novo, basta subir o .xls na pasta dados/sefaz do repositório.

REPO = 'Renata-Nascimento/Projeto-U02-CD'
PASTA_SEFAZ = 'dados/sefaz'
os.makedirs(PASTA_SEFAZ, exist_ok=True)

resposta = requests.get(f'https://api.github.com/repos/{REPO}/contents/{PASTA_SEFAZ}', timeout=60)
resposta.raise_for_status()

for item in resposta.json():
    if item['name'].lower().endswith('.xls'):
        destino = os.path.join(PASTA_SEFAZ, item['name'])
        if not os.path.exists(destino):          # se já baixou nesta sessão, não baixa de novo
            conteudo = requests.get(item['download_url'], timeout=120).content
            with open(destino, 'wb') as f:
                f.write(conteudo)

arquivos = sorted(glob.glob(f'{PASTA_SEFAZ}/*.xls'))
if not arquivos:
    raise FileNotFoundError(f'Nenhuma planilha .xls encontrada em {REPO}/{PASTA_SEFAZ}')
print(len(arquivos), 'arquivos')
for a in arquivos:
    print(' ', os.path.basename(a))

### Como é o arquivo bruto
É um relatório feito para impressão: 8 linhas de cabeçalho, municípios nas linhas, meses nas colunas,
linha de TOTAL e rodapé com observações.

In [ ]:
pd.read_excel(arquivos[0], sheet_name='ICMS', header=None).iloc[6:14, :5]

,0,1,2,3,4
6,NaN,NaN,NaN,NaN,NaN
7,NaN,ÍNDICE %,2 0 2 0,2 0 2 0,2 0 2 0
8,MUNICÍPIOS,2020,JANEIRO,FEVEREIRO,MARÇO
9,NaN,NaN,131462850.71,115681907.56,127462339.81
10,ACARI,0.2203,289612.660114,254847.242355,280799.534601
11,AFONSO BEZERRA,0.1961,257798.650242,226852.220725,249953.648367
12,AGUA NOVA,0.1064,139876.473155,123085.549644,135619.929558
13,ALEXANDRIA,0.1886,247938.936439,218176.077658,240393.972882


### Leitura de todas as planilhas

O layout muda de um ano para outro (colunas extras, mês repetido, TOTAL em posições diferentes). Por isso:
1. achamos a linha de cabeçalho pela palavra `MUNICÍPIOS`;
2. achamos as colunas pelo **nome do mês**;
3. ficamos só com as linhas cujo nome bate com a lista do IBGE (descarta totais e rodapés);
4. paramos na linha TOTAL que vem depois dos municípios.

In [ ]:
MESES = ['JANEIRO', 'FEVEREIRO', 'MARCO', 'ABRIL', 'MAIO', 'JUNHO',
         'JULHO', 'AGOSTO', 'SETEMBRO', 'OUTUBRO', 'NOVEMBRO', 'DEZEMBRO']

def ler_aba(arquivo, aba, ano):
    bruto = pd.read_excel(arquivo, sheet_name=aba, header=None)
    lin_cab = bruto.index[bruto[0].astype(str).str.upper().str.startswith('MUNIC')][0]
    cabecalho = [chave(c) for c in bruto.iloc[lin_cab]]
    corpo = bruto.iloc[lin_cab + 1:]

    k = corpo[0].map(chave).replace(DE_PARA)
    inicio = k[k.isin(list(COD_POR_CHAVE))].index[0]
    fim = corpo.index[(corpo.index > inicio) & corpo[0].astype(str).str.upper().str.contains('TOTA')]
    if len(fim):
        corpo, k = corpo.loc[:fim[0] - 1], k.loc[:fim[0] - 1]
    corpo = corpo.assign(cod_ibge=k.map(COD_POR_CHAVE)).dropna(subset=['cod_ibge'])

    colunas_meses = [j for j, c in enumerate(cabecalho) if c in MESES]
    valores = corpo[colunas_meses].apply(pd.to_numeric, errors='coerce')
    return pd.DataFrame({'cod_ibge': corpo['cod_ibge'].astype(int).values,
                         'ano': ano, 'tipo': aba,
                         'valor': valores.sum(axis=1).values})

partes = []
for arq in arquivos:
    ano = int(re.search(r'(20\d{2})', os.path.basename(arq)).group(1))   # 1º ano que aparece no nome do arquivo
    for aba in ['ICMS', 'IPVA', 'IPI', 'ROYALTIES']:
        partes.append(ler_aba(arq, aba, ano))
repasses = pd.concat(partes, ignore_index=True)
repasses = repasses[repasses['ano'].isin(ANOS)]

# conferência: cada ano × tipo deve ter os 167 municípios
repasses.pivot_table(index='ano', columns='tipo', values='cod_ibge', aggfunc='nunique')

tipo,ICMS,IPI,IPVA,ROYALTIES
ano,,,,
2020,167,167,167,167


In [ ]:
# Total repassado por ano (R$ milhões, valores nominais, sem correção)
(repasses.pivot_table(index='ano', columns='tipo', values='valor', aggfunc='sum') / 1e6).round(1)

tipo,ICMS,IPI,IPVA,ROYALTIES
ano,,,,
2020,1414.5,1.5,197.8,20.2


## 4. IBGE: população, PIB e inflação (API SIDRA)

- **População:** Censo 2022 (tabela 4709), usada como base fixa para todos os anos.
- **PIB municipal** 2020–2023 (tabela 5938), em mil reais, valores nominais.
- **IPCA** (tabela 1737): número-índice mensal. Usamos a média de cada ano para levar os valores a preços de 2023.

In [ ]:
def sidra(caminho):
    dados = requests.get('https://apisidra.ibge.gov.br/values' + caminho, timeout=120).json()
    return pd.DataFrame(dados[1:])

populacao = sidra('/t/4709/n6/in%20n3%2024/v/93/p/2022')[['D1C', 'V']]
populacao.columns = ['cod_ibge', 'populacao']
populacao = populacao.astype({'cod_ibge': int, 'populacao': int})

pib = sidra('/t/5938/n6/in%20n3%2024/v/37/p/' + ','.join(map(str, ANOS)))[['D1C', 'D3N', 'V']]
pib.columns = ['cod_ibge', 'ano', 'pib_mil_reais']
pib = pib.astype({'cod_ibge': int, 'ano': int, 'pib_mil_reais': float})

print(len(populacao), 'municípios com população')
pib.pivot_table(index='ano', values='cod_ibge', aggfunc='nunique').rename(columns={'cod_ibge': 'municípios com PIB'})

167 municípios com população


,municípios com PIB
ano,
2020,167
2021,167
2022,167
2023,167


In [ ]:
ipca = sidra(f'/t/1737/n1/all/v/2266/p/{ANOS[0]}01-{ANO_PRECOS}12')[['D3C', 'V']]
ipca['ano'] = ipca['D3C'].str[:4].astype(int)
ipca['V'] = ipca['V'].astype(float)
ipca = ipca.groupby('ano')['V'].mean()

# fator que leva R$ de cada ano para R$ de 2023
fator_2023 = (ipca[ANO_PRECOS] / ipca).rename('fator_ipca').reset_index()
fator_2023

,ano,fator_ipca
0,2020,1.237887
1,2021,1.143000
2,2022,1.045936
3,2023,1.000000


## 5. CadÚnico (MDS)

API pública do MDS (VIS DATA). Usamos a posição de **dezembro** de cada ano.
Atenção: o MDS usa o código IBGE com **6 dígitos** (sem o último dígito verificador).

In [ ]:
params = {
    'q': '*:*',
    'fq': ['sigla_uf:RN', 'anomes_s:(' + ' OR '.join(f'{a}12' for a in ANOS) + ')'],
    'fl': 'codigo_ibge,anomes_s,cadun_qtd_pessoas_cadastradas_i,cadun_qtd_pessoas_cadastradas_rfpc_ate_meio_sm_i',
    'rows': 2000, 'wt': 'json',
}
r = requests.get('https://aplicacoes.mds.gov.br/sagi/servicos/misocial', params=params, timeout=120)

cadunico = pd.DataFrame(r.json()['response']['docs']).rename(columns={
    'codigo_ibge': 'cod_ibge6',
    'cadun_qtd_pessoas_cadastradas_i': 'cad_pessoas',
    'cadun_qtd_pessoas_cadastradas_rfpc_ate_meio_sm_i': 'cad_baixa_renda',
})
cadunico['cod_ibge6'] = cadunico['cod_ibge6'].astype(int)
cadunico['ano'] = cadunico['anomes_s'].str[:4].astype(int)
cadunico = cadunico.drop(columns='anomes_s')

cadunico.pivot_table(index='ano', values='cod_ibge6', aggfunc='nunique').rename(columns={'cod_ibge6': 'municípios'})

,municípios
ano,
2020,167
2021,167
2022,167
2023,167


## 6. Integração: uma linha por município e ano (167 × 4 = 668 linhas)

In [ ]:
rep_anual = (repasses
             .pivot_table(index=['cod_ibge', 'ano'], columns='tipo', values='valor', aggfunc='sum')
             .rename(columns=str.lower).add_prefix('rep_').reset_index())
rep_anual.columns.name = None
rep_anual['rep_total'] = rep_anual[['rep_icms', 'rep_ipva', 'rep_ipi', 'rep_royalties']].sum(axis=1)

base = (municipios
        .merge(pd.DataFrame({'ano': ANOS}), how='cross')        # todas as combinações município × ano
        .merge(rep_anual, on=['cod_ibge', 'ano'], how='left')
        .merge(populacao, on='cod_ibge', how='left')            # Censo 2022 para todos os anos
        .merge(pib, on=['cod_ibge', 'ano'], how='left')
        .merge(fator_2023, on='ano', how='left')
        .assign(cod_ibge6=lambda d: d['cod_ibge'] // 10)
        .merge(cadunico, on=['cod_ibge6', 'ano'], how='left')
        .drop(columns='cod_ibge6'))

# valores reais (R$ de 2023)
base['rep_total_real'] = base['rep_total'] * base['fator_ipca']
base['pib_real_mil'] = base['pib_mil_reais'] * base['fator_ipca']

# variáveis por habitante
base['rep_per_capita'] = base['rep_total_real'] / base['populacao']
base['pib_per_capita'] = base['pib_real_mil'] * 1000 / base['populacao']
base['pct_baixa_renda'] = base['cad_baixa_renda'] / base['populacao'] * 100
base['porte'] = pd.cut(base['populacao'], bins=[0, 5_000, 10_000, 20_000, 50_000, np.inf],
                       labels=['até 5 mil', '5–10 mil', '10–20 mil', '20–50 mil', 'mais de 50 mil'])

print(base.shape)
base.head()

(668, 21)


,cod_ibge,municipio,mesorregiao,regiao_intermediaria,ano,rep_icms,rep_ipi,rep_ipva,rep_royalties,rep_total,...,pib_mil_reais,fator_ipca,cad_baixa_renda,cad_pessoas,rep_total_real,pib_real_mil,rep_per_capita,pib_per_capita,pct_baixa_renda,porte
0,2400109,Acari,Central Potiguar,Caicó,2020,3.116225e+06,3279.713210,385048.780,44590.552676,3.549144e+06,...,132249.0,1.237887,5656,6675,4.393441e+06,1.637094e+05,414.592918,15448.653756,53.373596,10–20 mil
1,2400109,Acari,Central Potiguar,Caicó,2021,NaN,NaN,NaN,NaN,NaN,...,141992.0,1.143000,5851,6940,NaN,1.622968e+05,NaN,15315.352846,55.213740,10–20 mil
2,2400109,Acari,Central Potiguar,Caicó,2022,NaN,NaN,NaN,NaN,NaN,...,173700.0,1.045936,6052,7262,NaN,1.816790e+05,NaN,17144.382242,57.110503,10–20 mil
3,2400109,Acari,Central Potiguar,Caicó,2023,NaN,NaN,NaN,NaN,NaN,...,195828.0,1.000000,5048,7041,NaN,1.958280e+05,NaN,18479.569690,47.636123,10–20 mil
4,2400208,Assú,Oeste Potiguar,Mossoró,2020,1.939472e+07,20412.232329,2109941.855,277522.046181,2.180259e+07,...,1169390.0,1.237887,30759,34268,2.698916e+07,1.447573e+06,477.718030,25622.579695,54.444562,mais de 50 mil


## 7. Qualidade dos dados

In [ ]:
print('Linhas:', len(base), '(esperado 167 × 4 =', 167 * len(ANOS), ')')
print('Nulos no total:', base.isna().sum().sum())
print('Pares município-ano duplicados:', base.duplicated(['cod_ibge', 'ano']).sum())

Linhas: 668 (esperado 167 × 4 = 668 )
Nulos no total: 3507
Pares município-ano duplicados: 0


In [ ]:
# Municípios com mais pessoas no CadÚnico do que habitantes (cadastro desatualizado ou população subestimada)
acima = base[base['cad_pessoas'] > base['populacao']]
print(len(acima), 'casos em', acima['municipio'].nunique(), 'municípios')
acima.pivot_table(index='municipio', columns='ano', values='cad_pessoas', aggfunc='sum')

31 casos em 12 municípios


ano,2020,2021,2022,2023
municipio,,,,
Bodó,2601.0,2704.0,2804.0,2843.0
Caiçara do Rio do Vento,NaN,NaN,3310.0,NaN
Japi,5652.0,5876.0,6050.0,5867.0
João Dias,NaN,NaN,2166.0,2171.0
Lagoa Salgada,NaN,NaN,9236.0,9688.0
Lagoa de Velhos,NaN,2595.0,2624.0,NaN
Monte das Gameleiras,2416.0,2540.0,2636.0,2645.0
Paraná,NaN,NaN,NaN,3976.0
Severiano Melo,NaN,NaN,5659.0,5718.0


In [ ]:
# Efeito da correção pela inflação no total do estado (R$ milhões)
efeito = base.groupby('ano')[['rep_total', 'rep_total_real']].sum() / 1e6
efeito['crescimento_nominal_%'] = (efeito['rep_total'] / efeito['rep_total'].iloc[0] - 1) * 100
efeito['crescimento_real_%'] = (efeito['rep_total_real'] / efeito['rep_total_real'].iloc[0] - 1) * 100
efeito.round(1)

,rep_total,rep_total_real,crescimento_nominal_%,crescimento_real_%
ano,,,,
2020,1634.1,2022.8,0.0,0.0
2021,0.0,0.0,-100.0,-100.0
2022,0.0,0.0,-100.0,-100.0
2023,0.0,0.0,-100.0,-100.0


In [ ]:
base[['rep_per_capita', 'pib_per_capita', 'pct_baixa_renda', 'populacao']].describe().round(1)

,rep_per_capita,pib_per_capita,pct_baixa_renda,populacao
count,167.0,668.0,668.0,668.0
mean,731.1,24008.0,65.9,19776.8
std,1176.5,22987.2,12.2,64679.2
min,123.5,10502.7,27.4,1822.0
25%,358.5,13889.8,58.5,4274.0
50%,476.4,16558.2,66.2,7834.0
75%,702.6,22556.0,72.4,12484.0
max,13342.2,190006.7,109.2,751300.0


## 8. Dicionário de dados

In [ ]:
dicionario = pd.DataFrame([
    ['cod_ibge', 'Código IBGE do município (chave de integração)', 'IBGE', '-'],
    ['municipio', 'Nome oficial do município', 'IBGE', '-'],
    ['mesorregiao', 'Mesorregião (Oeste, Central, Agreste, Leste Potiguar)', 'IBGE', '-'],
    ['regiao_intermediaria', 'Região intermediária (Natal, Mossoró, Caicó, Pau dos Ferros)', 'IBGE', '-'],
    ['ano', 'Ano de referência (2020 a 2023)', '-', 'ano'],
    ['rep_icms', 'Cota-parte do ICMS recebida no ano', 'SEFAZ/RN', 'R$ nominais'],
    ['rep_ipva', 'Cota-parte do IPVA recebida no ano', 'SEFAZ/RN', 'R$ nominais'],
    ['rep_ipi', 'Cota-parte do IPI-Exportação recebida no ano', 'SEFAZ/RN', 'R$ nominais'],
    ['rep_royalties', 'Royalties do petróleo repassados no ano', 'SEFAZ/RN', 'R$ nominais'],
    ['rep_total', 'Soma dos 4 repasses', 'Derivada', 'R$ nominais'],
    ['populacao', 'População residente — Censo 2022 (fixa para todos os anos)', 'IBGE (SIDRA 4709)', 'pessoas'],
    ['pib_mil_reais', 'PIB a preços correntes', 'IBGE (SIDRA 5938)', 'R$ mil nominais'],
    ['fator_ipca', 'Fator que converte R$ do ano para R$ de 2023 (média anual do IPCA)', 'IBGE (SIDRA 1737)', 'razão'],
    ['cad_pessoas', 'Pessoas inscritas no CadÚnico — dezembro', 'MDS', 'pessoas'],
    ['cad_baixa_renda', 'Pessoas no CadÚnico com renda até ½ salário mínimo per capita — dezembro', 'MDS', 'pessoas'],
    ['rep_total_real', 'Repasse total corrigido pelo IPCA', 'Derivada', 'R$ de 2023'],
    ['pib_real_mil', 'PIB corrigido pelo IPCA', 'Derivada', 'R$ mil de 2023'],
    ['rep_per_capita', 'Repasse total real por habitante', 'Derivada', 'R$ de 2023/hab'],
    ['pib_per_capita', 'PIB real por habitante', 'Derivada', 'R$ de 2023/hab'],
    ['pct_baixa_renda', 'Pessoas de baixa renda no CadÚnico em % da população', 'Derivada', '%'],
    ['porte', 'Faixa de população (Censo 2022)', 'Derivada', 'categoria'],
], columns=['variável', 'descrição', 'fonte', 'unidade'])

dicionario['tipo'] = dicionario['variável'].map(base.dtypes.astype(str))
dicionario

,variável,descrição,fonte,unidade,tipo
0,cod_ibge,Código IBGE do município (chave de integração),IBGE,-,int64
1,municipio,Nome oficial do município,IBGE,-,object
2,mesorregiao,"Mesorregião (Oeste, Central, Agreste, Leste Po...",IBGE,-,object
3,regiao_intermediaria,"Região intermediária (Natal, Mossoró, Caicó, P...",IBGE,-,object
4,ano,Ano de referência (2020 a 2023),-,ano,int64
5,rep_icms,Cota-parte do ICMS recebida no ano,SEFAZ/RN,R$ nominais,float64
6,rep_ipva,Cota-parte do IPVA recebida no ano,SEFAZ/RN,R$ nominais,float64
7,rep_ipi,Cota-parte do IPI-Exportação recebida no ano,SEFAZ/RN,R$ nominais,float64
8,rep_royalties,Royalties do petróleo repassados no ano,SEFAZ/RN,R$ nominais,float64
9,rep_total,Soma dos 4 repasses,Derivada,R$ nominais,float64


## 9. Salvar

In [ ]:
base.to_csv('saida/base_municipios_2020_2023.csv', index=False)
repasses.to_csv('saida/repasses_2020_2023.csv', index=False)
dicionario.to_csv('saida/dicionario_de_dados.csv', index=False)

## 10. Viabilidade, qualidade e aprendizados

**Viabilidade:** as 3 fontes foram coletadas e integradas para os **167 municípios em todos os 4 anos**,
sem dados faltando (668 linhas).

**Problemas encontrados e como tratamos:**
1. As planilhas da SEFAZ são relatórios, não tabelas, e o formato muda a cada ano. → leitura pelo nome do mês e filtro pela lista do IBGE.
2. Não há código IBGE e há 15 municípios escritos de formas diferentes (inclusive nomes antigos). → chave padronizada + de-para.
3. Um filtro para apagar rodapés com a palavra "LEI" apagava *Monte das Game**lei**ras* e *Tabo**lei**ro Grande*. → aprendizado: filtrar pelo que é válido (lista do IBGE), não pelo que parece lixo.
4. O PIB municipal só vai até 2023 e a população tem quebra de série (Censo 2022 ~7% abaixo das estimativas) e falta 2023. → período 2020–2023 e população fixa no Censo 2022.
5. Os valores de anos diferentes não são comparáveis por causa da inflação. → correção pelo IPCA, a preços de 2023.
6. Alguns municípios têm mais pessoas no CadÚnico do que habitantes (cadastro desatualizado). → manter, mas tratar como outlier na EDA.

**Primeiro indício (a confirmar no CP2):** os municípios que mais recebem por habitante são pequenos e têm alta
atividade econômica (petróleo, eólicas, sal), e não necessariamente os mais pobres. Faz sentido, porque a regra do
ICMS distribui principalmente pela atividade econômica gerada no município.